# Session 6: Network Devices, Components and Packet Tracer

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hamnakhan123/Cybersecurity-and-Digital-Forensics/blob/main/term-1/03-computer-networks/code/session-06-network-devices-packet-tracer.ipynb)

*Cybersecurity and Digital Forensics* · Prepared by **Hamna Zahid Khan**, Junior Lecturer

Run each cell from top to bottom (Shift+Enter). Every example is self-contained and safe to run in a lab or on Colab.

## `session-06_switch_vs_hub.py`

In [1]:
"""Session 6 - Why switches replaced hubs: simulate MAC-address learning.

A hub repeats every frame out of every port. A switch learns which MAC lives on
which port from the SOURCE address of each frame, then forwards only to the
right port (flooding only while it does not know the destination yet).
Run:  python session-06_switch_vs_hub.py
"""

BROADCAST = "ff:ff:ff:ff:ff:ff"


class Hub:
    def __init__(self, ports): self.ports = ports
    def receive(self, in_port, src, dst):
        return [p for p in range(1, self.ports + 1) if p != in_port]


class Switch:
    def __init__(self, ports):
        self.ports = ports
        self.cam = {}                       # MAC address table: mac -> port

    def receive(self, in_port, src, dst):
        self.cam[src] = in_port             # 1. learn from the source address
        if dst != BROADCAST and dst in self.cam:
            out = self.cam[dst]             # 2. known unicast -> one port
            return [] if out == in_port else [out]
        return [p for p in range(1, self.ports + 1) if p != in_port]   # 3. flood


if __name__ == "__main__":
    pcs = {"A": ("aa:00:00:00:00:0a", 1), "B": ("aa:00:00:00:00:0b", 2),
           "C": ("aa:00:00:00:00:0c", 3), "D": ("aa:00:00:00:00:0d", 4)}
    traffic = [("A", "B"), ("B", "A"), ("C", "A"), ("A", "C"), ("D", "BCAST"), ("A", "B")]
    for dev in (Hub(4), Switch(4)):
        total = 0
        print(f"\n{type(dev).__name__}:")
        for s, d in traffic:
            src, port = pcs[s]
            dst = BROADCAST if d == "BCAST" else pcs[d][0]
            outs = dev.receive(port, src, dst)
            total += len(outs)
            print(f"  {s} -> {d:<5} in port {port}  sent out ports {outs}")
        print(f"  frames copied onto cables: {total}")
        if isinstance(dev, Switch):
            print("  learned MAC table:", {m[-2:]: p for m, p in dev.cam.items()})


Hub:
  A -> B     in port 1  sent out ports [2, 3, 4]
  B -> A     in port 2  sent out ports [1, 3, 4]
  C -> A     in port 3  sent out ports [1, 2, 4]
  A -> C     in port 1  sent out ports [2, 3, 4]
  D -> BCAST in port 4  sent out ports [1, 2, 3]
  A -> B     in port 1  sent out ports [2, 3, 4]
  frames copied onto cables: 18

Switch:
  A -> B     in port 1  sent out ports [2, 3, 4]
  B -> A     in port 2  sent out ports [1]
  C -> A     in port 3  sent out ports [1]
  A -> C     in port 1  sent out ports [3]
  D -> BCAST in port 4  sent out ports [1, 2, 3]
  A -> B     in port 1  sent out ports [2]
  frames copied onto cables: 10
  learned MAC table: {'0a': 1, '0b': 2, '0c': 3, '0d': 4}


## `session-06_routing_table.py`

In [2]:
"""Session 6 - How a router picks a route: longest-prefix match, then
administrative distance, then metric.

Run:  python session-06_routing_table.py
"""
import ipaddress

# (prefix, next hop / interface, source, administrative distance, metric)
TABLE = [
    ("0.0.0.0/0",      "203.0.113.1 (ISP)",  "static", 1,   0),
    ("10.0.0.0/8",     "10.255.0.2",         "OSPF",   110, 30),
    ("10.1.0.0/16",    "10.255.0.6",         "OSPF",   110, 20),
    ("10.1.5.0/24",    "Gi0/1 (connected)",  "connected", 0, 0),
    ("10.1.5.0/24",    "10.255.0.9",         "RIP",    120, 2),
    ("172.16.0.0/12",  "10.255.0.2",         "EIGRP",  90,  28160),
    ("172.16.8.0/22",  "10.255.0.6",         "RIP",    120, 3),
]


def lookup(dst: str):
    ip = ipaddress.ip_address(dst)
    matches = [r for r in TABLE if ip in ipaddress.ip_network(r[0])]
    longest = max(ipaddress.ip_network(r[0]).prefixlen for r in matches)
    candidates = [r for r in matches if ipaddress.ip_network(r[0]).prefixlen == longest]
    best = min(candidates, key=lambda r: (r[3], r[4]))      # lower AD, then lower metric
    return best, matches


if __name__ == "__main__":
    print("Routing table:")
    for p, nh, src, ad, m in TABLE:
        print(f"  {p:<16} via {nh:<20} {src:<9} [AD {ad}/metric {m}]")
    for dst in ["10.1.5.20", "10.1.9.9", "10.200.1.1", "172.16.9.4", "172.20.1.1", "8.8.8.8"]:
        best, matches = lookup(dst)
        print(f"\n{dst}: {len(matches)} matching routes -> use {best[0]} via {best[1]} [{best[2]}]")

Routing table:
  0.0.0.0/0        via 203.0.113.1 (ISP)    static    [AD 1/metric 0]
  10.0.0.0/8       via 10.255.0.2           OSPF      [AD 110/metric 30]
  10.1.0.0/16      via 10.255.0.6           OSPF      [AD 110/metric 20]
  10.1.5.0/24      via Gi0/1 (connected)    connected [AD 0/metric 0]
  10.1.5.0/24      via 10.255.0.9           RIP       [AD 120/metric 2]
  172.16.0.0/12    via 10.255.0.2           EIGRP     [AD 90/metric 28160]
  172.16.8.0/22    via 10.255.0.6           RIP       [AD 120/metric 3]

10.1.5.20: 5 matching routes -> use 10.1.5.0/24 via Gi0/1 (connected) [connected]

10.1.9.9: 3 matching routes -> use 10.1.0.0/16 via 10.255.0.6 [OSPF]

10.200.1.1: 2 matching routes -> use 10.0.0.0/8 via 10.255.0.2 [OSPF]

172.16.9.4: 3 matching routes -> use 172.16.8.0/22 via 10.255.0.6 [RIP]

172.20.1.1: 2 matching routes -> use 172.16.0.0/12 via 10.255.0.2 [EIGRP]

8.8.8.8: 1 matching routes -> use 0.0.0.0/0 via 203.0.113.1 (ISP) [static]
